In [ ]:
# Notebook 04 – Evaluation Report
# Purpose: Visualize and analyze performance metrics from model experiments

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# === Load the metrics dataset ===
df = pd.read_csv("results/metrics.csv")
sns.set_style("whitegrid")

# Ensure correct numeric format for metrics
df["F1"] = pd.to_numeric(df["F1"], errors="coerce")
df["ROC_AUC"] = pd.to_numeric(df["ROC_AUC"], errors="coerce")

# Combine Sampling and Weighting for compact visualization
df["Sampling_Weighting"] = df["Sampling"].astype(str) + "_" + df["Weighting"].astype(str)

# === 1. F1 Score Distribution across Models & Sampling ===
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x="Model", y="F1", hue="Sampling")
plt.title("F1 Score Distribution by Model and Sampling")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("results/plots/f1_boxplot_by_model_sampling.png")
plt.show()

# === 2. Aggregate Mean Metrics by Model + Strategy ===
agg_df = df.groupby(['Model', 'Sampling', 'Weighting']).agg({
    'F1': 'mean',
    'ROC_AUC': 'mean'
}).reset_index()

# Display top 5 rows as preview
print("Top mean F1 scores by combination:")
print(agg_df.sort_values("F1", ascending=False).head(5))

# === 3. Heatmap of F1 Scores by Model and Sampling+Weighting ===
heatmap_data = df.groupby(["Model", "Sampling_Weighting"])["F1"].mean().unstack()

plt.figure(figsize=(12, 6))
sns.heatmap(heatmap_data, annot=True, fmt=".2f", cmap="YlGnBu")
plt.title("F1 Score Heatmap by Model and Sampling+Weighting")
plt.ylabel("Model")
plt.xlabel("Sampling + Weighting")
plt.tight_layout()
plt.savefig("results/plots/f1_heatmap.png")
plt.show()

# === 4. Scatter Plot: ROC_AUC vs F1 Score Tradeoff ===
plt.figure(figsize=(8, 6))
sns.scatterplot(data=agg_df, x="ROC_AUC", y="F1", hue="Model", style="Sampling", s=100)
plt.title("F1 vs ROC-AUC Tradeoff (Averaged Across Runs)")
plt.xlabel("ROC-AUC")
plt.ylabel("F1 Score")
plt.grid(True)
plt.tight_layout()
plt.savefig("results/plots/f1_vs_rocauc_scatter.png")
plt.show()

# === 5. Top 5 F1 Combinations ===
top_f1 = agg_df.sort_values("F1", ascending=False).head(5)
print("\nTop 5 model combinations by F1:")
print(top_f1)

# === 6. Metric Correlation ===
plt.figure(figsize=(5, 4))
sns.heatmap(df[["F1", "ROC_AUC"]].corr(), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Metric Correlation (F1 vs ROC_AUC)")
plt.tight_layout()
plt.savefig("results/plots/metric_correlation_heatmap.png")
plt.show()

print("\nEvaluation analysis complete. Plots saved to results/plots/")
